In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")
df['ClaimOccurred'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df.head()


C:\Users\HP\AppData\Local\Temp\ipykernel_18192\784722658.py:5: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,ClaimOccurred,Margin
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0,21.929825
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0,21.929825
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,0,0.000000
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0,512.848070
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,0,0.000000


In [4]:
contingency_table = pd.crosstab(df['Gender'], df['ClaimOccurred'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print("CHI-SQUARE TEST RESULTS")
print("-" * 30)
print(f"Chi-Square Statistic (χ²): {chi2:.4f}")
print(f"P-Value: {p_value:.6f}")
print(f"Degrees of Freedom: {dof}")

if p_value < 0.05:
    print("\nConclusion: There IS a statistically significant relationship between Gender and Claim Occurrence.")
else:
    print("\nConclusion: There is NO statistically significant relationship between Gender and Claim Occurrence.")


CHI-SQUARE TEST RESULTS
------------------------------
Chi-Square Statistic (χ²): 7.2559
P-Value: 0.026570
Degrees of Freedom: 2

Conclusion: There IS a statistically significant relationship between Gender and Claim Occurrence.


In [5]:
contingency_table_cover = pd.crosstab(df['CoverType'], df['ClaimOccurred'])
chi2_c, p_value_c, dof_c, expected_c = stats.chi2_contingency(contingency_table_cover)

print("CHI-SQUARE TEST RESULTS (CoverType vs Claim Occurrence)")
print("-" * 50)
print(f"Chi-Square Statistic (χ²): {chi2_c:.4f}")
print(f"P-Value: {p_value_c:.6f}")
print(f"Degrees of Freedom: {dof_c}")

if p_value_c < 0.05:
    print("\nConclusion: There IS a statistically significant relationship between CoverType and Claim Occurrence.")
else:
    print("\nConclusion: There is NO statistically significant relationship between CoverType and Claim Occurrence.")


CHI-SQUARE TEST RESULTS (CoverType vs Claim Occurrence)
--------------------------------------------------
Chi-Square Statistic (χ²): 8172.5782
P-Value: 0.000000
Degrees of Freedom: 21

Conclusion: There IS a statistically significant relationship between CoverType and Claim Occurrence.


In [6]:
contingency_table_vehicle = pd.crosstab(df['VehicleType'], df['ClaimOccurred'])
chi2_v, p_value_v, dof_v, expected_v = stats.chi2_contingency(contingency_table_vehicle)

print("CHI-SQUARE TEST RESULTS (VehicleType vs Claim Occurrence)")
print("-" * 55)
print(f"Chi-Square Statistic (χ²): {chi2_v:.4f}")
print(f"P-Value: {p_value_v:.6f}")
print(f"Degrees of Freedom: {dof_v}")

if p_value_v < 0.05:
    print("\nConclusion: There IS a statistically significant relationship between VehicleType and Claim Occurrence.")
else:
    print("\nConclusion: There is NO statistically significant relationship between VehicleType and Claim Occurrence.")


CHI-SQUARE TEST RESULTS (VehicleType vs Claim Occurrence)
-------------------------------------------------------
Chi-Square Statistic (χ²): 1.5864
P-Value: 0.811230
Degrees of Freedom: 4

Conclusion: There is NO statistically significant relationship between VehicleType and Claim Occurrence.


In [8]:
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

cont_prov = pd.crosstab(df['Province'], df['ClaimOccurred'])
chi2_p, p_prov_freq, dof_p, exp_p = stats.chi2_contingency(cont_prov)
print("CLAIM FREQUENCY (Province)")
print("-"*40)
print(f"Chi-Square: {chi2_p:.4f} | p-value: {p_prov_freq:.6g} | df: {dof_p}")
if p_prov_freq < 0.05:
    print("Conclusion: Reject H0 — Claim frequency differs across provinces.\n")
else:
    print("Conclusion: Fail to reject H0 — No evidence of frequency differences across provinces.\n")

claims = df[df['TotalClaims'] > 0]
groups = [grp['TotalClaims'].values for name, grp in claims.groupby('Province') if len(grp) >= 5]  # require at least 5 obs
province_names = [name for name, grp in claims.groupby('Province') if len(grp) >= 5]

print("CLAIM SEVERITY (Province) — Kruskal-Wallis (non-parametric)")
print("-"*40)
if len(groups) >= 2:
    kw_stat, kw_p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H: {kw_stat:.4f} | p-value: {kw_p:.6g}")
    if kw_p < 0.05:
        print("Conclusion: Reject H0 — Claim severity differs across provinces.")
        try:
            import numpy as np
            claims['logClaims'] = np.log1p(claims['TotalClaims'])
            tuk = pairwise_tukeyhsd(claims['logClaims'], claims['Province'])
            print("\nTukey HSD (sample):")
            print(tuk.summary().data[:10])  # print a few lines to avoid huge output
        except Exception as e:
            pass
    else:
        print("Conclusion: Fail to reject H0 — No evidence of severity differences across provinces.")
else:
    print("Not enough provinces with claims to perform Kruskal-Wallis reliably.\n")


print("\nMARGIN (Province) — Kruskal-Wallis")
print("-"*40)
groups_margin = [grp['Margin'].values for name, grp in df.groupby('Province') if len(grp) >= 5]
if len(groups_margin) >= 2:
    kw_m_stat, kw_m_p = stats.kruskal(*groups_margin)
    print(f"Kruskal-Wallis H: {kw_m_stat:.4f} | p-value: {kw_m_p:.6g}")
    if kw_m_p < 0.05:
        print("Conclusion: Reject H0 — Margin differs across provinces.")
    else:
        print("Conclusion: Fail to reject H0 — No evidence of margin differences across provinces.")
else:
    print("Not enough provinces to perform Kruskal-Wallis on Margin.\n")


CLAIM FREQUENCY (Province)
----------------------------------------
Chi-Square: 104.1909 | p-value: 5.92551e-19 | df: 8
Conclusion: Reject H0 — Claim frequency differs across provinces.

CLAIM SEVERITY (Province) — Kruskal-Wallis (non-parametric)
----------------------------------------
Kruskal-Wallis H: 106.0927 | p-value: 2.41466e-19
Conclusion: Reject H0 — Claim severity differs across provinces.

Tukey HSD (sample):
[['group1', 'group2', 'meandiff', 'p-adj', 'lower', 'upper', 'reject'], ['Eastern Cape', 'Free State', np.float64(0.2836), np.float64(0.9998), np.float64(-1.3815), np.float64(1.9487), np.False_], ['Eastern Cape', 'Gauteng', np.float64(0.0613), np.float64(1.0), np.float64(-0.6591), np.float64(0.7816), np.False_], ['Eastern Cape', 'KwaZulu-Natal', np.float64(0.3992), np.float64(0.7661), np.float64(-0.3436), np.float64(1.142), np.False_], ['Eastern Cape', 'Limpopo', np.float64(-0.6753), np.float64(0.3777), np.float64(-1.6097), np.float64(0.2591), np.False_], ['Eastern Cape